# **Week 1: Setup**

In [ ]:
# Core environment and package imports
from pathlib import Path
import os
import sys
import platform
import random
import json

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, log_loss
import scipy
import statsmodels
import statsmodels.api as sm

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('✅ Core imports successful')
print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')
print(f'matplotlib: {matplotlib.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'SciPy: {scipy.__version__}')
print(f'statsmodels: {statsmodels.__version__}')

# Optional GitHub clone helper — safe to leave blank
GITHUB_REPO_URL = ''  # Example: 'https://github.com/USERNAME/Against-All-Odds.git'

def running_in_colab():
    return 'google.colab' in sys.modules

if GITHUB_REPO_URL.strip():
    repo_name = GITHUB_REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    target = Path('/content') / repo_name if running_in_colab() else Path.cwd() / repo_name
    if target.exists():
        print(f'✅ Repository already exists at: {target}')
    else:
        import subprocess
        subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(target)], check=True)
        print(f'✅ Repository cloned to: {target}')
else:
    print('ℹ️ GitHub clone skipped. This is expected because GITHUB_REPO_URL is blank.')

PROJECT_TITLE = 'Against All Odds'
PROJECT_START = '2026-08-29'
PROJECT_END = '2026-12-03'
REQUIRED_SPORTS = ['NBA', 'NFL', 'Premier League']
OPTIONAL_EXTENSIONS = ['La Liga', 'Bundesliga', 'Serie A', 'Ligue 1']
RELIABILITY_LABELS = ['Predict', 'Caution', 'Abstain']

MAIN_RESEARCH_QUESTION = (
    'Can pre-game statistical information be used to predict professional sports outcomes '
    'while also identifying which predictions are reliable, fragile, or unusually vulnerable to an upset?'
)

PROJECT_CONFIG = {
    'title': PROJECT_TITLE,
    'start_date': PROJECT_START,
    'end_date': PROJECT_END,
    'required_sports': REQUIRED_SPORTS,
    'optional_extensions': OPTIONAL_EXTENSIONS,
    'reliability_labels': RELIABILITY_LABELS,
    'random_seed': RANDOM_SEED,
}

print(json.dumps(PROJECT_CONFIG, indent=2))
print('\nMain research question:')
print(MAIN_RESEARCH_QUESTION)

# Choose a safe project root for Colab or local execution
CURRENT_DIR = Path.cwd().resolve()
KNOWN_REPO_NAMES = {'Against-All-Odds', 'Against_All_Odds', 'Against All Odds'}

if CURRENT_DIR.name in KNOWN_REPO_NAMES:
    PROJECT_ROOT = CURRENT_DIR
elif running_in_colab():
    PROJECT_ROOT = Path('/content/Against-All-Odds')
else:
    PROJECT_ROOT = CURRENT_DIR / 'Against-All-Odds'

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'✅ Project root: {PROJECT_ROOT}')

# Create the Week 1 project directory structure
DIRECTORIES = [
    'data/raw/nba', 'data/raw/nfl', 'data/raw/soccer',
    'data/clean/nba', 'data/clean/nfl', 'data/clean/soccer',
    'data/model_ready/nba', 'data/model_ready/nfl', 'data/model_ready/soccer',
    'notebooks',
    'src/collection', 'src/cleaning', 'src/features', 'src/modeling', 'src/evaluation',
    'figures', 'results', 'docs', 'report', 'presentation'
]

created = []
for relative_dir in DIRECTORIES:
    path = PROJECT_ROOT / relative_dir
    path.mkdir(parents=True, exist_ok=True)
    created.append(path)

print(f'✅ Verified {len(created)} project directories')
for path in created:
    print(' -', path.relative_to(PROJECT_ROOT))

DATA_DICTIONARY_COLUMNS = [
    'variable_name', 'sport', 'level', 'category', 'data_type', 'description',
    'source', 'pre_game_only', 'calculation_or_definition', 'missing_value_rule',
    'required_or_optional', 'notes'
]

starter_rows = [
    {
        'variable_name': 'game_id', 'sport': 'ALL', 'level': 'game', 'category': 'identifier',
        'data_type': 'string', 'description': 'Unique game or match identifier', 'source': 'TBD',
        'pre_game_only': 'yes', 'calculation_or_definition': 'Source ID or reproducible composite key',
        'missing_value_rule': 'Must not be missing', 'required_or_optional': 'required', 'notes': 'Primary key'
    },
    {
        'variable_name': 'game_date', 'sport': 'ALL', 'level': 'game', 'category': 'time',
        'data_type': 'date', 'description': 'Date of the game or match', 'source': 'TBD',
        'pre_game_only': 'yes', 'calculation_or_definition': 'Convert to consistent datetime format',
        'missing_value_rule': 'Must not be missing', 'required_or_optional': 'required', 'notes': 'Used for chronological validation'
    },
    {
        'variable_name': 'elo_diff', 'sport': 'ALL', 'level': 'game', 'category': 'team_strength',
        'data_type': 'float', 'description': 'Pre-game home Elo minus away Elo', 'source': 'Derived',
        'pre_game_only': 'yes', 'calculation_or_definition': 'home_pre_game_elo - away_pre_game_elo',
        'missing_value_rule': 'Document handling', 'required_or_optional': 'required',
        'notes': 'Calculate before current-game Elo update'
    },
    {
        'variable_name': 'fragility_score', 'sport': 'ALL', 'level': 'game', 'category': 'reliability',
        'data_type': 'float', 'description': 'Final prediction fragility score', 'source': 'Derived',
        'pre_game_only': 'yes', 'calculation_or_definition': 'Defined after perturbation experiments',
        'missing_value_rule': 'Not available in early weeks', 'required_or_optional': 'required_final',
        'notes': 'Do not invent a final formula during Week 1'
    },
    {
        'variable_name': 'reliability_label', 'sport': 'ALL', 'level': 'game', 'category': 'reliability',
        'data_type': 'string', 'description': 'Predict, Caution, or Abstain', 'source': 'Derived',
        'pre_game_only': 'yes', 'calculation_or_definition': 'Thresholds selected using validation data',
        'missing_value_rule': 'Not available in early weeks', 'required_or_optional': 'required_final',
        'notes': 'Avoid arbitrary final thresholds in Week 1'
    },
]

data_dictionary = pd.DataFrame(starter_rows, columns=DATA_DICTIONARY_COLUMNS)
data_dictionary_path = PROJECT_ROOT / 'docs' / 'data_dictionary_template.csv'
data_dictionary.to_csv(data_dictionary_path, index=False)

print(f'✅ Data dictionary saved: {data_dictionary_path}')
display(data_dictionary)

# Synthetic leakage-safety demonstration
demo = pd.DataFrame({
    'game_number': [1, 2, 3, 4, 5, 6],
    'team_points': [100, 110, 105, 120, 98, 115],
})

demo['points_last_3_pre_game'] = demo['team_points'].shift(1).rolling(window=3, min_periods=1).mean()

# At game 4, the rolling value should use games 1-3 only.
expected_game4 = np.mean([100, 110, 105])
actual_game4 = demo.loc[3, 'points_last_3_pre_game']
assert np.isclose(actual_game4, expected_game4), 'Leakage-safe rolling feature test failed'

print('✅ Leakage-safe rolling feature test passed')
display(demo)

# 1) Package/version smoke test
required_modules = {
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
    'scikit-learn': sklearn.__version__,
    'scipy': scipy.__version__,
    'statsmodels': statsmodels.__version__,
}
assert all(required_modules.values())
print('✅ Package smoke test passed')

# 2) pandas / NumPy smoke test
test_df = pd.DataFrame({
    'x1': [0.1, 0.2, 0.3, 0.8, 0.9, 1.0],
    'x2': [1.0, 0.9, 0.8, 0.3, 0.2, 0.1],
    'y':  [0,   0,   0,   1,   1,   1],
})
assert test_df.shape == (6, 3)
assert np.isfinite(test_df[['x1', 'x2']].to_numpy()).all()
print('✅ pandas / NumPy smoke test passed')

# 3) scikit-learn smoke test
X = test_df[['x1', 'x2']]
y = test_df['y']
clf = LogisticRegression(random_state=RANDOM_SEED).fit(X, y)
probs = clf.predict_proba(X)[:, 1]
preds = clf.predict(X)
assert len(probs) == len(test_df)
assert 0.0 <= accuracy_score(y, preds) <= 1.0
print('✅ scikit-learn smoke test passed')

# 4) statsmodels smoke test
X_sm = sm.add_constant(X)
sm_model = sm.Logit(y, X_sm).fit(disp=False)
assert len(sm_model.params) == X_sm.shape[1]
print('✅ statsmodels smoke test passed')

# 5) matplotlib smoke test
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot([1, 2, 3], [1, 4, 9])
ax.set_title('Against All Odds setup test')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.close(fig)
print('✅ matplotlib smoke test passed')

# 6) file read/write smoke test
smoke_path = PROJECT_ROOT / 'results' / '_setup_smoke_test.csv'
test_df.to_csv(smoke_path, index=False)
reloaded = pd.read_csv(smoke_path)
assert reloaded.equals(test_df)
smoke_path.unlink()
print('✅ file read/write smoke test passed')

# Final Week 1 setup status
required_paths = [PROJECT_ROOT / d for d in DIRECTORIES]
missing_paths = [str(p) for p in required_paths if not p.exists()]

if missing_paths:
    raise RuntimeError(f'Missing required project directories: {missing_paths}')

summary = {
    'project': PROJECT_TITLE,
    'project_root': str(PROJECT_ROOT),
    'required_sports': REQUIRED_SPORTS,
    'directories_verified': len(required_paths),
    'data_dictionary_exists': data_dictionary_path.exists(),
    'setup_status': 'PASS',
}

print('=' * 70)
print('✅ AGAINST ALL ODDS — WEEK 1 SETUP PASSED')
print('=' * 70)
for key, value in summary.items():
    print(f'{key}: {value}')
print('\nNext major task: Week 2 data-source testing and season selection.')


✅ Core imports successful
Python: 3.13.15
pandas: 2.2.3
NumPy: 2.1.3
matplotlib: 3.10.0
scikit-learn: 1.6.1
SciPy: 1.16.3
statsmodels: 0.14.6
ℹ️ GitHub clone skipped. This is expected because GITHUB_REPO_URL is blank.
{
  "title": "Against All Odds",
  "start_date": "2026-08-29",
  "end_date": "2026-12-03",
  "required_sports": [
    "NBA",
    "NFL",
    "Premier League"
  ],
  "optional_extensions": [
    "La Liga",
    "Bundesliga",
    "Serie A",
    "Ligue 1"
  ],
  "reliability_labels": [
    "Predict",
    "Caution",
    "Abstain"
  ],
  "random_seed": 42
}

Main research question:
Can pre-game statistical information be used to predict professional sports outcomes while also identifying which predictions are reliable, fragile, or unusually vulnerable to an upset?
✅ Project root: /content/Against-All-Odds
✅ Verified 20 project directories
 - data/raw/nba
 - data/raw/nfl
 - data/raw/soccer
 - data/clean/nba
 - data/clean/nfl
 - data/clean/soccer
 - data/model_ready/nba
 - data/mo

,variable_name,sport,level,category,data_type,description,source,pre_game_only,calculation_or_definition,missing_value_rule,required_or_optional,notes
0,game_id,ALL,game,identifier,string,Unique game or match identifier,TBD,yes,Source ID or reproducible composite key,Must not be missing,required,Primary key
1,game_date,ALL,game,time,date,Date of the game or match,TBD,yes,Convert to consistent datetime format,Must not be missing,required,Used for chronological validation
2,elo_diff,ALL,game,team_strength,float,Pre-game home Elo minus away Elo,Derived,yes,home_pre_game_elo - away_pre_game_elo,Document handling,required,Calculate before current-game Elo update
3,fragility_score,ALL,game,reliability,float,Final prediction fragility score,Derived,yes,Defined after perturbation experiments,Not available in early weeks,required_final,Do not invent a final formula during Week 1
4,reliability_label,ALL,game,reliability,string,"Predict, Caution, or Abstain",Derived,yes,Thresholds selected using validation data,Not available in early weeks,required_final,Avoid arbitrary final thresholds in Week 1


✅ Leakage-safe rolling feature test passed


,game_number,team_points,points_last_3_pre_game
0,1,100,NaN
1,2,110,100.000000
2,3,105,105.000000
3,4,120,105.000000
4,5,98,111.666667
5,6,115,107.666667


✅ Package smoke test passed
✅ pandas / NumPy smoke test passed
✅ scikit-learn smoke test passed
✅ statsmodels smoke test passed
✅ matplotlib smoke test passed
✅ file read/write smoke test passed
✅ AGAINST ALL ODDS — WEEK 1 SETUP PASSED
project: Against All Odds
project_root: /content/Against-All-Odds
required_sports: ['NBA', 'NFL', 'Premier League']
directories_verified: 20
data_dictionary_exists: True
setup_status: PASS

Next major task: Week 2 data-source testing and season selection.


/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.13/dist-packages/statsmodels/discrete/discrete_model.py:227: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
/usr/local/lib/python3.13/dist-packa